# Extract Patch of 224x224 around each ROI Centre Coordinate

In [ ]:
import pandas as pd
import numpy as np

import os
import glob

import tifffile as tiff
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2

In [2]:
# Increase the image pixel limit
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

In [3]:

OME_LEVEL = 2 # 10x magnification
DOWNSAMPLE = 2**OME_LEVEL
#root_path = 'C:/Users/hooll/Dropbox/19 PostGrad/00 CB Lab/01 projects/06 MultiModal/data/LEAP'
root_path = 'C:/Users/hooll/Dropbox/19 PostGrad/00 CB Lab/01 projects/06 MultiModal/data'
input_path = 'E:/01-data/LEAP/WSI/2-aligned/10x'
#input_path = os.path.join(root_path,'01-raw','images','HnE','aligned')
output_path = os.path.join(root_path,'01-raw','images','HnE','patches')


In [4]:
sample_metadata = pd.read_excel(os.path.join(root_path,'01-raw','metadata',"meta_data.xlsx"), index_col=0)

In [5]:
sample_metadata = sample_metadata[sample_metadata['Sample_Type'] != 'PELLET']

In [6]:
sample_metadata['Sample_Type'].value_counts(normalize=True)

Sample_Type
DIAGNOSTIC    0.812009
RESECTION     0.158775
Pre-NACT      0.015515
1_1           0.010478
2_2           0.003224
Name: proportion, dtype: float64

# Calculate Downsampled ROI Coordinates

In [7]:
# Downsample coordinates by a factor of 2 (20x) - 10x=4
sample_metadata.loc[:, "ROICoordinateX_downsampled"] = (sample_metadata["ROICoordinateX"] / DOWNSAMPLE).astype(int)
sample_metadata.loc[:, "ROICoordinateY_downsampled"] = (sample_metadata["ROICoordinateY"] / DOWNSAMPLE).astype(int)

# Set the LEAP ID

In [8]:
from LEAPdata import get_slide_id, get_leap_ids

In [9]:
LEAPIDs = [
    "LEAP088", "LEAP091", "LEAP092", "LEAP093", "LEAP095", "LEAP103", "LEAP105", "LEAP106",
    "LEAP110", "LEAP111", "LEAP112", "LEAP113", "LEAP115", "LEAP118", "LEAP119", "LEAP120",
    "LEAP121", "LEAP138", "LEAP139", "LEAP142", "LEAP144", "LEAP145", "LEAP146", "LEAP148"
]


In [10]:
len(LEAPIDs)

24

# Load the Metadata
Need this to get the ROI Centre Coordinates for each slide

In [11]:
subset_metadata = sample_metadata[sample_metadata["LEAP_ID"].isin(LEAPIDs)]

## Get Unique ROI Coordinates

In [12]:
# Drop duplicates based on Scan.Name, and Roi
unique_rows_df = subset_metadata.drop_duplicates(subset=[ "LEAP_ID","Tissue_ID", "Roi","Segment"])

In [13]:

result_df = unique_rows_df[[
    "DCC_ID","Segment","LEAP_ID","Tissue_ID","Roi", 
    "ROICoordinateX_downsampled", "ROICoordinateY_downsampled"
]]
result_df = result_df.set_index('DCC_ID')
result_df.head()



,Segment,LEAP_ID,Tissue_ID,Roi,ROICoordinateX_downsampled,ROICoordinateY_downsampled
DCC_ID,,,,,,
DSP-1001660018886-C-A02.dcc,PanCK+,LEAP088,LEAP088_B,19.0,9068,2773
DSP-1001660018886-C-A03.dcc,CD45+,LEAP088,LEAP088_B,20.0,8604,3498
DSP-1001660018886-C-A04.dcc,PanCK+,LEAP088,LEAP088_B,20.0,8604,3498
DSP-1001660018886-C-A05.dcc,CD45+,LEAP088,LEAP088_B,21.0,8604,3797
DSP-1001660018886-C-A06.dcc,PanCK+,LEAP088,LEAP088_B,21.0,8604,3797


# Extract H&E patch around ROI

The results_df ROI corresponds to roi_dict[i]['name'] (not the roi_id)

X and Y are round the wrong way

In [14]:
for seg in subset_metadata.Segment.unique():
    os.makedirs(os.path.join(output_path,seg),exist_ok=True)

In [15]:
def extract_patches(img, df, output_path):

    size = 224
    half = int(224/2)


    heimg_h, heimg_w,_ = img.shape

    for index, row in df.iterrows():
        ## need to switch the ROI coordinate system
        roi_x = row['ROICoordinateY_downsampled']
        roi_y = row['ROICoordinateX_downsampled']

        ## use max and min to make sure we don't go outside the edge of the image - shouldn't be needed but good practice
        x_start = max(roi_x - half, 0)
        y_start = max(roi_y - half, 0)
        x_end = min(roi_x + half, heimg_w)
        y_end = min(roi_y + half, heimg_h)

        tissue = row['Tissue_ID'].replace('_', '-')
        leapid = row['LEAP_ID'].replace('_', '-')
        region = img[y_start:y_end, x_start:x_end]
        
        outfilename = f"{index}_{leapid}_{row['Segment']}_ROI{int(row['Roi'])}_{roi_x}_{roi_y}.png"
        #print(f"{outfilename}: {x_start}, {y_start} to {x_end}, {y_end}")
        if region is None: 
            print(f"{outfilename}: region is NONE")
        elif region.size == 0:
            print(f"{outfilename}: region is 0")
        else: 
            cv2.imwrite(os.path.join(output_path,row['Segment'],outfilename),region)
        


In [16]:
for LEAPID in LEAPIDs:
    SLIDEID = get_slide_id(LEAPID)
    print(f"{LEAPID} and {SLIDEID}")
    wsi_name=f"{LEAPID}_{SLIDEID.lower()}-aligned.tiff"
    # Load H&E
    he_path = os.path.join(input_path,wsi_name) 
    #print(he_path)
    he_img = Image.open(he_path)
    he_img = np.array(he_img)
    df= result_df[result_df['LEAP_ID']==LEAPID]
    extract_patches(he_img, df, output_path)
    


LEAP088 and Slide_38
DSP-1001660018887-D-D08.dcc_LEAP088_CD45+_ROI1_3442_18628.png: region is 0
DSP-1001660018887-D-D09.dcc_LEAP088_PanCK+_ROI1_3442_18628.png: region is 0
DSP-1001660018887-D-D10.dcc_LEAP088_CD45+_ROI2_3740_18628.png: region is 0
DSP-1001660018887-D-D11.dcc_LEAP088_PanCK+_ROI2_3740_18628.png: region is 0
DSP-1001660018887-D-D12.dcc_LEAP088_CD45+_ROI3_3442_18925.png: region is 0
DSP-1001660018887-D-E01.dcc_LEAP088_PanCK+_ROI3_3442_18925.png: region is 0
DSP-1001660018887-D-E02.dcc_LEAP088_CD45+_ROI4_3740_18925.png: region is 0
DSP-1001660018887-D-E03.dcc_LEAP088_PanCK+_ROI4_3740_18925.png: region is 0
DSP-1001660018887-D-E04.dcc_LEAP088_CD45+_ROI5_4076_18719.png: region is 0
DSP-1001660018887-D-E05.dcc_LEAP088_PanCK+_ROI5_4076_18719.png: region is 0
DSP-1001660018887-D-E06.dcc_LEAP088_CD45+_ROI6_4362_18706.png: region is 0
DSP-1001660018887-D-E07.dcc_LEAP088_PanCK+_ROI6_4362_18706.png: region is 0
DSP-1001660018887-D-E08.dcc_LEAP088_CD45+_ROI7_4063_19004.png: region is 

# END